## Setup

In [1]:
from googleapiclient.discovery import build
import json
import re
from collections import Counter
from datetime import datetime
from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import glob
import os

In [2]:
with open ("../performative.txt", "r") as f:
    lines = f.readlines()
    API_KEY = lines[2].strip()
    
yt = build("youtube", "v3", developerKey=API_KEY)

### Static/Maps

In [3]:
GENRE_MAP = {
    "1":  "Film & Animation",
    "2":  "Autos & Vehicles",
    "10": "Music",
    "15": "Pets & Animals",
    "17": "Sports",
    "19": "Travel & Events",
    "20": "Gaming",
    "22": "People & Blogs",
    "23": "Comedy",
    "24": "Entertainment",
    "25": "News & Politics",
    "26": "Howto & Style",
    "27": "Education",
    "28": "Science & Technology",
    "29": "Nonprofits & Activism",
}

In [4]:
QUERIES = [
    "video essay film analysis",
    "philosophy explained documentary",
    "internet culture deep dive",
    "history documentary essay channel",
    "game design analysis",
]
MAX_PER_QUERY = 50  # how many channels to collect per keyword/phrase


## Scripts

In [5]:
def search_channels_queries(query, max_results, seen):
    collected = []
    next_page = None

    while len(collected) < max_results:
        resp = yt.search().list(
            part="snippet",
            q = query, 
            type="channel",
            relevanceLanguage = "en",
            maxResults=50,
            pageToken = next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"]["channelId"]
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   item["snippet"]["channelId"],
                    "channel_name": item["snippet"]["title"],
                })
            if len(collected) >= max_results:
                break
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

In [12]:
def search_channels_playlist(playlist_id, seen):
    collected = []
    next_page = None

    while True:
        resp = yt.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"].get("videoOwnerChannelId")  # dict[key] raises a KeyError when the key is missing. dict.get(key) gracefully returns None. verify?
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   cid,
                    "channel_name": item["snippet"].get("videoOwnerChannelTitle"),
                })
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

## Running

In [13]:
all_channels = []
seen = set()

for query in QUERIES:
    print(f"Searching: {query}")
    results = search_channels_queries(query, MAX_PER_QUERY, seen)
    print(f" got {len(results)} channels")
    all_channels.extend(results)

with open("../data/processed/ve_channels/ve_channels_queries.json", "r") as f:
    existing = json.load(f)
combined = existing + all_channels
with open("../data/processed/ve_channels_queries.json", "w") as f:
    json.dump(all_channels, f, indent=2)

Searching: video essay film analysis
 got 50 channels
Searching: philosophy explained documentary
 got 24 channels
Searching: internet culture deep dive
 got 50 channels
Searching: history documentary essay channel
 got 50 channels
Searching: game design analysis
 got 50 channels


In [14]:
seen = set()

results = search_channels_playlist("PLXz6Pf8rae1mIGQ8IdpggWP2RmW1wmXsa", seen)

with open("../data/processed/ve_channels/ve_channels_playlist.json", "w") as f:
    json.dump(results, f, indent=2)

In [15]:
results2 = search_channels_playlist("PLbOF9TQmQrNgWHaLCPaE81l4hEL-2Rrjf", set())

with open("../data/processed/ve_channels/ve_channels_playlist_2.json", "w") as f:
    json.dump(results2, f, indent=2)

## Viewing Data

In [16]:
with open("../data/processed/ve_channels/ve_channels_queries.json", "r") as f:
    data_queries = json.load(f)

In [17]:
df_queries = pd.DataFrame(data_queries)
df_queries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    225 non-null    object
 1   channel_name  225 non-null    object
dtypes: object(2)
memory usage: 3.6+ KB


In [18]:
with open("../data/processed/ve_channels/ve_channels_playlist.json", "r") as f:
    data_playlist = json.load(f)

In [19]:
df_playlist = pd.DataFrame(data_playlist)
df_playlist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    106 non-null    object
 1   channel_name  106 non-null    object
dtypes: object(2)
memory usage: 1.8+ KB


In [20]:
with open("../data/processed/ve_channels/ve_channels_playlist_2.json", "r") as f:
    data_playlist2 = json.load(f)

In [21]:
df_playlist2 = pd.DataFrame(data_playlist2)
df_playlist2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    23 non-null     object
 1   channel_name  23 non-null     object
dtypes: object(2)
memory usage: 500.0+ bytes


## Cleaning Data

In [39]:
df = pd.concat([df_queries, df_playlist, df_playlist2])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 355 entries, 0 to 22
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    354 non-null    object
 1   channel_name  354 non-null    object
dtypes: object(2)
memory usage: 8.3+ KB


In [40]:
df = df.drop_duplicates(subset="channel_id").reset_index(drop=True)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 341 entries, 0 to 340
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    340 non-null    object
 1   channel_name  340 non-null    object
dtypes: object(2)
memory usage: 5.5+ KB


In [42]:
rows_to_add = [{"channel_id": "UCD6TPU-PvTMvqgzC_AM7_uA", "channel_name": "The People Profiles"},
               {"channel_id": "UC1DTYW241WD64ah5BFWn4JA", "channel_name": "Sam O'Nella Academy"},]

In [43]:
df[df["channel_id"] == "UC1DTYW241WD64ah5BFWn4JA"]

,channel_id,channel_name


In [44]:
df.iloc[50:100]

,channel_id,channel_name
50,UCwbDCAPugfe1KrbCZ9F0VPw,Philosophy Origins
51,UCTNKkxZDNDjVGcibLg1D6Ww,Philosophy Matters
52,UCCfI-j18jSgT45svMcofILw,Documentaries and Philosophy
53,UCp1mRTkVlqDnxz_9S0YD9YQ,Philosophies for Life
54,UCIuCDa-yyDddjzVfu6WMiNQ,Philosophy Documentaries
55,UCBz-uxcltpYVrL9On42z2qQ,Philosophy Corner
56,UCVvux6AY30Iqe7bxxcjL2LA,Sam's Philosophy
57,UCXRNtFb3LCHxaV4u0XlrAvQ,Philosophy Sleeping Story
58,UC7IcJI8PUf5Z3zKxnZvTBog,The School of Life
59,UCWTc-B9-9j5Tykq7ou4tL3Q,Philosophy Decoded


In [45]:
df.iloc[100:150]

,channel_id,channel_name
100,UCt3JiNkefsfbA2N4SgEkoiQ,Visual Venture
101,UCnoYaHN1zDCD-vcNXdwF-8g,Deep Dive
102,UCY6DpwQetygtye7GM1zVHmw,tiffanyferg
103,UCGKUQte8RBCNnI_OYpckErg,Hannah Alonzo
104,UCmdRI2gLXb8hp2cNcvuj0MQ,Danielle
105,UCvCzQh2gf1zantUDmvKP_8A,Viral Deep Dive
106,UCa7Niyr46r2QHVdhIB_-_3A,TheDigitalRabbitHole
107,UCSUwFKuCzI42rOaIEIFM7Xw,Based Internet Mysteries
108,UC6QuQhhRicNwoxtuFzq_Qdw,Doomscroll Dive
109,UCDpOXvDm7WStFeSDRxnzdqA,Just Happen It


In [46]:
rows_to_drop = [11, 12, 18, 36, 50, 51, 52, 54,
                 56, 57, 59, 60, 62, 63, 65, 66, 67,
                 69, 70, 72, 73, 74, 77, 78, 79, 80, 82, 
                 83, 84, 86, 87, 88, 89, 90, 91, 92, 94, 97,
                 98, 105, 106, 107, 108, 109, 110, 111, 112,
                 113, 114, 115, 116, 117, 118, 119, 121, 122,
                 123, 124, 125, 126, 127, 128, 129, 130, 131,
                 132, 134, 136, 137, 138, 139, 141, 142, 143,
                 144, 145, 146, 148, 149, 172, 178, 186, 187,
                 188, 189, 191, 198, 216, 224]

In [47]:
df.iloc[150:200]

,channel_id,channel_name
150,UCTv4kbT56qnD3iL-o-pfEiw,Historical Amnesia
151,UCkS_HP3m9NXOgswVAKbMeJQ,Then & Now
152,UCrx2zrPjhGRi9TwszZiLwEg,Horses
153,UCT6Y5JJPKe_JDMivpKgVXew,Fall of Civilizations
154,UCmDTX6KHkuPS_L63lyK2BbQ,History of Elections
155,UC_3pplzbKMZsP5zBH_6SVJQ,Chris Chan: A Comprehensive History
156,UCz8BwZBpbZzHH7xQrvt0qRQ,DJBoHistory Tv
157,UCLXo7UDZvByw2ixzpQCufnA,Vox
158,UC-R24vaD9ZoucOlhYkIZtDw,The Decline and fall of British Sci-Fi
159,UCmGSJVG3mCRXVOP4yZrU1Dw,Johnny Harris


In [48]:
df.iloc[200:250]

,channel_id,channel_name
200,UCNaAppb_FDGLH3vLnUcY3Pg,Design Frame
201,UCO9k7tfA_H9FZ_sDi6bNFXw,Games Every Designer Should Play
202,UCZMF14eNxvuReRTceX_mbqQ,The Game Overanalyser
203,UCE5FiTkmvVHzVScukla75uw,Game Designer Plays
204,UCtZMWqGbT2UDdw1gGqVEpRg,Let's Talk Game Design
205,UCpkmLZ83NKt6FTN4kKUgdyg,yakkocmn
206,UCv1DvRY5PyHHt3KN9ghunuw,Masahiro Sakurai on Creating Games
207,UCJr7DrVt9FEQjys6mWP8Y1w,Designing For
208,UCSdma21fnJzgmPodhC9SJ3g,NakeyJakey
209,UC1_U9dYmPupJTQUz4N3UBGw,Out Of Nickels


In [49]:
df = df.drop(rows_to_drop)
df = df.reset_index(drop=True)
df.head()

,channel_id,channel_name
0,UCqgh7QNlVV8kv3LBTJZYWrg,The Film Essay
1,UCjFqcJQXGZ6T6sxyFB-5i6A,Every Frame a Painting
2,UCr8r7UVsDaRiIQSVUHVBd_A,Nikki Carreon
3,UCjKSoJlPgcK6BmoSqXpj5xQ,Action Button
4,UCUyvQV2JsICeLZP4c_h40kA,Thomas Flight


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    251 non-null    object
 1   channel_name  251 non-null    object
dtypes: object(2)
memory usage: 4.1+ KB


In [52]:
df.to_json("../data/processed/ve_channels/ve.json")